# Cognix Validation Experiment (Round 1)

**Goal:** Test whether brain-predicted similarity (via TRIBE v2) diverges from standard semantic similarity.

**Requirements:** A100 GPU (40GB+ VRAM), HuggingFace token with LLaMA 3.2 access.

**Decision gate:**
- r > 0.95 → project likely not viable
- r = 0.8-0.95 → weak signal, investigate outliers
- r < 0.8 → meaningful divergence, proceed to Round 2

## 0. Setup

**Before running:** Add your HuggingFace token as a Colab secret:
1. Click the **key icon** in the left sidebar
2. Add new secret: Name = `HF_TOKEN`, Value = your token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Toggle notebook access **ON**

Then **Runtime > Change runtime type > A100 GPU + High RAM**

Now run every cell top to bottom.

In [ ]:
# ---- System dependencies ----
!apt-get update -qq && apt-get install -y -qq ffmpeg

# ---- Install dependencies ----
!pip uninstall -y numpy
!pip install 'numpy>=1.26.4,<2.1.0'

!git clone https://github.com/facebookresearch/tribev2.git
%cd tribev2
!pip install -e '.[plotting]'
%cd ..

!pip install sentence-transformers scipy scikit-learn matplotlib tqdm
!pip install --upgrade torchcodec

# ---- Clone Cognix repo for validation pairs ----
!git clone https://github.com/gdavid7/cognix.git

# ---- HuggingFace auth (reads from Colab secrets) ----
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
import json
import os
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy.spatial.distance import cosine as cosine_dist
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

## 1. Load validation pairs

In [ ]:
PAIRS_PATH = "cognix/data/validation_pairs.jsonl"

pairs = []
with open(PAIRS_PATH) as f:
    for line in f:
        pairs.append(json.loads(line))

print(f"Loaded {len(pairs)} pairs")
categories = {}
for p in pairs:
    cat = p["category"]
    categories[cat] = categories.get(cat, 0) + 1
print("Categories:", categories)

## 2. Compute semantic similarity (sentence-transformer)

This is lightweight and can run on CPU.

In [ ]:
st_model = SentenceTransformer("all-MiniLM-L6-v2")

semantic_sims = []
for p in tqdm(pairs, desc="Semantic similarity"):
    emb_a = st_model.encode(p["text_a"])
    emb_b = st_model.encode(p["text_b"])
    sim = 1.0 - cosine_dist(emb_a, emb_b)
    semantic_sims.append(sim)

semantic_sims = np.array(semantic_sims)
print(f"Semantic similarity: mean={semantic_sims.mean():.3f}, std={semantic_sims.std():.3f}")

## 3. Compute brain similarity (TRIBE v2)

**This requires GPU.** Each text takes ~5-10 seconds on A100.

In [ ]:
from tribev2 import TribeModel

tribe_model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
print("TRIBE v2 loaded.")

In [ ]:
def text_to_brain_vector(text: str) -> np.ndarray:
    """Run TRIBE v2 on text and return mean-pooled brain vector (20484,)."""
    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
        tmp_path = f.name
    try:
        events = tribe_model.get_events_dataframe(text_path=tmp_path)
        preds, _segments = tribe_model.predict(events=events)
        brain_tensor = np.asarray(preds)  # (T, 20484)
        return brain_tensor.mean(axis=0)  # (20484,)
    finally:
        os.unlink(tmp_path)

In [ ]:
# Collect all unique texts to avoid duplicate TRIBE runs
unique_texts = set()
for p in pairs:
    unique_texts.add(p["text_a"])
    unique_texts.add(p["text_b"])
unique_texts = list(unique_texts)
print(f"{len(unique_texts)} unique texts to process")

In [ ]:
# Run TRIBE on all unique texts and cache results
brain_vectors = {}
for text in tqdm(unique_texts, desc="TRIBE inference"):
    try:
        brain_vectors[text] = text_to_brain_vector(text)
    except Exception as e:
        print(f"FAILED on text: {text[:60]}... -> {e}")
        brain_vectors[text] = None

n_success = sum(1 for v in brain_vectors.values() if v is not None)
print(f"Successfully processed {n_success}/{len(unique_texts)} texts")

In [ ]:
# Compute brain similarity for each pair
brain_sims = []
valid_mask = []
for p in pairs:
    va = brain_vectors.get(p["text_a"])
    vb = brain_vectors.get(p["text_b"])
    if va is not None and vb is not None:
        sim = 1.0 - cosine_dist(va, vb)
        brain_sims.append(sim)
        valid_mask.append(True)
    else:
        brain_sims.append(np.nan)
        valid_mask.append(False)

brain_sims = np.array(brain_sims)
valid_mask = np.array(valid_mask)
print(f"Brain similarity computed for {valid_mask.sum()}/{len(pairs)} pairs")
print(f"Brain similarity: mean={np.nanmean(brain_sims):.3f}, std={np.nanstd(brain_sims):.3f}")

## 4. Save raw results (download these from Colab)

In [ ]:
results = []
for i, p in enumerate(pairs):
    results.append({
        "id": p["id"],
        "category": p["category"],
        "expected": p["expected"],
        "sim_semantic": float(semantic_sims[i]),
        "sim_brain": float(brain_sims[i]) if valid_mask[i] else None,
        "text_a_preview": p["text_a"][:80],
        "text_b_preview": p["text_b"][:80],
    })

with open("validation_results.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")

# Also save the brain vectors for later use
np.savez_compressed("brain_vectors_cache.npz", **{
    str(i): v for i, (text, v) in enumerate(brain_vectors.items()) if v is not None
})
# Save text-to-index mapping
with open("brain_vectors_index.json", "w") as f:
    json.dump({str(i): text for i, (text, v) in enumerate(brain_vectors.items()) if v is not None}, f)

print("Results saved to validation_results.jsonl")
print("Brain vectors saved to brain_vectors_cache.npz")

## 5. Correlation analysis (THE DECISION GATE)

In [ ]:
# Filter to valid pairs only
sem_valid = semantic_sims[valid_mask]
brain_valid = brain_sims[valid_mask]

pearson_r, pearson_p = stats.pearsonr(sem_valid, brain_valid)
spearman_r, spearman_p = stats.spearmanr(sem_valid, brain_valid)

print("=" * 60)
print("DECISION GATE: Semantic vs Brain Similarity Correlation")
print("=" * 60)
print(f"Pearson r:  {pearson_r:.4f}  (p={pearson_p:.2e})")
print(f"Spearman r: {spearman_r:.4f}  (p={spearman_p:.2e})")
print()

if pearson_r > 0.95:
    print("VERDICT: r > 0.95 — brain similarity ~ semantic similarity.")
    print("The project likely does not add meaningful value. Rethink the approach.")
elif pearson_r > 0.80:
    print("VERDICT: r = 0.8-0.95 — partial overlap with some divergence.")
    print("Investigate the divergent pairs below. Proceed with caution.")
else:
    print(f"VERDICT: r = {pearson_r:.2f} < 0.8 — meaningful divergence!")
    print("Brain similarity captures something different from semantic similarity.")
    print("Proceed to Round 2 (1000 pairs) and full system build.")

## 6. Scatter plot

In [ ]:
# Color by category
category_colors = {
    "paraphrase": "#2ecc71",
    "unrelated": "#95a5a6",
    "cognitive_load": "#e74c3c",
    "emotional_arousal": "#e67e22",
    "sensorimotor": "#9b59b6",
    "spatial_scene": "#3498db",
    "syntactic_surprise": "#1abc9c",
    "narrative_suspense": "#f39c12",
    "theory_of_mind": "#e91e63",
    "control_same_topic_diff_processing": "#607d8b",
}

fig, ax = plt.subplots(figsize=(10, 8))

for cat in category_colors:
    mask = np.array([p["category"] == cat and valid_mask[i] for i, p in enumerate(pairs)])
    if mask.sum() == 0:
        continue
    ax.scatter(
        semantic_sims[mask],
        brain_sims[mask],
        c=category_colors[cat],
        label=cat,
        alpha=0.7,
        s=60,
        edgecolors="white",
        linewidth=0.5,
    )

# Diagonal reference line (perfect correlation)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="y=x")

ax.set_xlabel("Semantic Similarity (sentence-transformer)", fontsize=12)
ax.set_ylabel("Brain Similarity (TRIBE v2, mean-pooled)", fontsize=12)
ax.set_title(f"Cognix Validation: Semantic vs Brain Similarity\nPearson r={pearson_r:.3f}, Spearman r={spearman_r:.3f}", fontsize=14)
ax.legend(loc="upper left", fontsize=8)
ax.set_xlim(-0.1, 1.05)
ax.set_ylim(-0.1, 1.05)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("validation_scatter.png", dpi=150)
plt.show()
print("Scatter plot saved to validation_scatter.png")

## 7. Per-category analysis

In [ ]:
print(f"{'Category':<40} {'N':>3}  {'Sem Mean':>8}  {'Brain Mean':>10}  {'Diff':>6}")
print("-" * 75)

for cat in sorted(category_colors.keys()):
    mask = np.array([p["category"] == cat and valid_mask[i] for i, p in enumerate(pairs)])
    if mask.sum() == 0:
        continue
    sem_mean = semantic_sims[mask].mean()
    brain_mean = brain_sims[mask].mean()
    diff = brain_mean - sem_mean
    print(f"{cat:<40} {mask.sum():>3}  {sem_mean:>8.3f}  {brain_mean:>10.3f}  {diff:>+6.3f}")

## 8. Most divergent pairs

Pairs where brain says "similar" but semantics says "not similar" — the key evidence.

In [ ]:
# Compute divergence: brain_sim - semantic_sim (positive = brain sees more similarity)
divergence = brain_sims - semantic_sims

# Get indices sorted by divergence (most brain-over-semantic first)
valid_indices = np.where(valid_mask)[0]
sorted_idx = valid_indices[np.argsort(divergence[valid_indices])[::-1]]

print("TOP 15 PAIRS: Brain says 'similar' but semantics says 'not similar'")
print("=" * 90)
for rank, idx in enumerate(sorted_idx[:15], 1):
    p = pairs[idx]
    print(f"\n#{rank} (id={p['id']}, category={p['category']})")
    print(f"  Semantic: {semantic_sims[idx]:.3f}  |  Brain: {brain_sims[idx]:.3f}  |  Divergence: {divergence[idx]:+.3f}")
    print(f"  A: {p['text_a'][:100]}...")
    print(f"  B: {p['text_b'][:100]}...")

In [ ]:
print("\nTOP 10 PAIRS: Semantics says 'similar' but brain says 'not similar'")
print("=" * 90)
for rank, idx in enumerate(sorted_idx[-10:], 1):
    p = pairs[idx]
    print(f"\n#{rank} (id={p['id']}, category={p['category']})")
    print(f"  Semantic: {semantic_sims[idx]:.3f}  |  Brain: {brain_sims[idx]:.3f}  |  Divergence: {divergence[idx]:+.3f}")
    print(f"  A: {p['text_a'][:100]}...")
    print(f"  B: {p['text_b'][:100]}...")

## 9. Summary and next steps

In [ ]:
print("VALIDATION EXPERIMENT SUMMARY")
print("=" * 60)
print(f"Total pairs: {len(pairs)}")
print(f"Valid pairs (both TRIBE succeeded): {valid_mask.sum()}")
print(f"Pearson correlation:  r={pearson_r:.4f}")
print(f"Spearman correlation: r={spearman_r:.4f}")
print()
print("NEXT STEPS:")
if pearson_r > 0.95:
    print("- The brain and semantic similarities are nearly identical.")
    print("- Consider: is the mean-pooling too aggressive? Try max-pool or region-specific pooling.")
    print("- Consider: are the divergence pairs actually divergent in TRIBE's representation?")
    print("- If nothing helps, the project may not produce a meaningfully different embedding.")
elif pearson_r > 0.80:
    print("- There is some divergence. Look at which categories diverge most.")
    print("- Scale up to 1000 pairs (Round 2) for statistical robustness.")
    print("- Focus the project's story on the categories with strongest divergence.")
else:
    print("- Strong divergence detected. The hypothesis holds.")
    print("- Scale to Round 2 (1000 pairs) to confirm.")
    print("- Then proceed to building the full Cognix pipeline (Phase 2+).")